To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i>
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install -qqq wandb  # Install WandB for logging

In [2]:
# ============================================
# CLEAR GPU MEMORY
# ============================================
import torch
import gc

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
# Force garbage collection
gc.collect()

print("GPU memory cleared!")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB total")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")


GPU memory cleared!
GPU Memory: 15.44 GB total
GPU Memory Allocated: 0.00 GB
GPU Memory Cached: 0.00 GB


### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/nhotin/anaconda3/envs/gptoss_finetune/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.444 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.28s/it]


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [5]:
from transformers import TextStreamer
import shutil
import os

# Clear compiled cache if it exists (fixes TorchRuntimeError)
cache_dir = "unsloth_compiled_cache"
if os.path.exists(cache_dir):
    try:
        shutil.rmtree(cache_dir)
        print("✅ Đã xóa compiled cache")
    except Exception as e:
        print(f"⚠️ Không thể xóa cache: {e}")

# Set model to inference mode
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "có mấy loại công ty?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

✅ Đã xóa compiled cache
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-06

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>có mấy loại công ty?<|end|><|start|>assistant<|channel|>analysis<|message|>User asks in Vietnamese: "có mấy loại công ty?" meaning "what types of companies exist?" They want categories of company types. Provide overview: sole proprietorship, partnership, limited liability company, corporation, etc.<|end|><|start|>assistant<|channel|>final<|message|>**Công ty ở Việt Nam thường được


Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [6]:
from transformers import TextStreamer

# Set model to inference mode
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "có mấy loại công ty?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-06

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>có mấy loại công ty?<|end|><|start|>assistant<|channel|>analysis<|message|>The user writes Vietnamese: "có mấy loại công ty?" asking what types of companies are there. They likely want to know about business company types: corporations (Ltd., Inc.), LLC, sole proprietorship, partnership, cooperative, etc. They want something about types of business entities in a legal


Lastly we will test it using `reasoning_effort` to `high`

In [7]:
from transformers import TextStreamer

# Set model to inference mode
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "có mấy loại công ty?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-06

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>có mấy loại công ty?<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "có mấy loại công ty?" In Vietnamese, that means "How many types of companies?" The user is presumably asking something: "What types of companies are there?" It's ambiguous. They might be asking about business types: for example, types of legal entities: corporations, limited


<a name="Data"></a>
### Data Prep

Dataset **Pháp luật Việt Nam** sẽ được sử dụng để fine-tune model GPT-OSS 20B. Dataset này chứa các cặp câu hỏi-trả lời về pháp luật Việt Nam, bao gồm:
- **Câu hỏi**: Các câu hỏi tư vấn pháp lý từ người dùng
- **Câu trả lời**: Câu trả lời chi tiết dựa trên các văn bản quy phạm pháp luật hiện hành
- **Tham chiếu**: Các điều luật và văn bản pháp luật liên quan
- **Loại câu hỏi**: Phân loại câu hỏi (query, consultation, etc.)

Dataset được lưu tại: `/home/nhotin/work/LegalBizAI_project/finetune/data_finetunning/qaset_article.json`

In [8]:
import json
from datasets import Dataset

# ============================================
# LOAD DATASET - Pháp luật Việt Nam
# ============================================
DATA_DIR = "/home/nhotin/work/LegalBizAI_project/finetune/data_finetunning"

print("Đang load dataset từ file JSON...")
with open(f"{DATA_DIR}/qaset_article.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print(f"Đã load {len(qa_data)} cặp câu hỏi-trả lời")

# ============================================
# CONVERT TO GPT-OSS HARMONY FORMAT
# ============================================
# GPT-OSS Harmony format yêu cầu:
# - developer role: Instructions cho model
# - user role: Câu hỏi
# - assistant role: Response với 2 channels (analysis và final)

def convert_to_gpt_oss_format(qa_item):
    """
    Chuyển đổi một cặp Q&A pháp luật thành format GPT-OSS Harmony
    với reasoning channels (analysis và final)
    """
    question = qa_item.get("question", "").strip()
    answer = qa_item.get("answer", "").strip()
    
    # Validate required fields
    if not question or not answer:
        return None
    
    references = qa_item.get("references", [])
    type_question = qa_item.get("type_question", "query")
    
    # Tạo phần context từ references nếu có
    context_text = ""
    if references:
        ref_texts = []
        for ref in references:
            if isinstance(ref, list) and len(ref) >= 2:
                ref_texts.append(f"{ref[0]} {ref[1]}")
        if ref_texts:
            context_text = f"\n\nCăn cứ pháp lý: {', '.join(ref_texts)}"
    
    # Tạo phần analysis (reasoning) - channel đầu tiên
    # Đây là phần "thinking" của model trước khi đưa ra câu trả lời
    analysis_text = (
        f"Câu hỏi này liên quan đến pháp luật Việt Nam. "
        f"Tôi cần phân tích để đưa ra câu trả lời chính xác dựa trên "
        f"các văn bản quy phạm pháp luật hiện hành.{context_text}\n\n"
        f"Đây là câu hỏi loại {type_question}, tôi sẽ trả lời dựa trên các quy định cụ thể."
    )
    
    # Tạo content cho assistant với 2 channels theo format GPT-OSS Harmony
    # Format: <|channel|>channel_name<|message|>content
    assistant_content = (
        f"<|channel|>analysis<|message|>{analysis_text}\n"
        f"<|channel|>final<|message|>{answer}"
    )
    
    # Tạo messages theo format GPT-OSS Harmony
    # GPT-OSS sử dụng role "developer" để set instructions
    messages = [
        {
            "role": "developer",
            "content": (
                "# Instructions\n\nreasoning language: Vietnamese\n\n"
                "Bạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. "
                "Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành. "
                "Câu trả lời phải chính xác, có căn cứ pháp lý rõ ràng và dễ hiểu."
            ),
        },
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_content},
    ]
    
    return {"messages": messages}


# Chuyển đổi tất cả dữ liệu
print("Đang chuyển đổi dữ liệu sang format GPT-OSS Harmony...")
converted_data = []
skipped = 0

for i, qa in enumerate(qa_data):
    converted = convert_to_gpt_oss_format(qa)
    if converted is None:
        skipped += 1
        if skipped <= 5:  # Show first 5 skipped items
            print(f"Bỏ qua item {i}: thiếu question hoặc answer")
    else:
        converted_data.append(converted)

if skipped > 0:
    print(f"Đã bỏ qua {skipped} mẫu không hợp lệ")

dataset = Dataset.from_list(converted_data)

print(f"Đã chuyển đổi {len(dataset)} mẫu dữ liệu sang format GPT-OSS Harmony")
print(f"\nVí dụ messages đầu tiên:")
print(json.dumps(dataset[0]["messages"], indent=2, ensure_ascii=False))

dataset

Đang load dataset từ file JSON...
Đã load 2539 cặp câu hỏi-trả lời
Đang chuyển đổi dữ liệu sang format GPT-OSS Harmony...
Đã chuyển đổi 2539 mẫu dữ liệu sang format GPT-OSS Harmony

Ví dụ messages đầu tiên:
[
  {
    "content": "# Instructions\n\nreasoning language: Vietnamese\n\nBạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành. Câu trả lời phải chính xác, có căn cứ pháp lý rõ ràng và dễ hiểu.",
    "role": "developer"
  },
  {
    "content": "Hội đồng giải thể doanh nghiệp do Nhà nước nắm giữ 100% vốn điều lệ có được quyền sử dụng con dấu của doanh nghiệp hay không?",
    "role": "user"
  },
  {
    "content": "<|channel|>analysis<|message|>Câu hỏi này liên quan đến pháp luật Việt Nam. Tôi cần phân tích để đưa ra câu trả lời chính xác dựa trên các văn bản quy phạm pháp luật hiện hành.\n\nCăn cứ pháp lý: Điều 44 Nghị định 23/2022/NĐ-CP\n\nĐây là câu hỏi loại query, tôi sẽ trả lời dựa trên các qu

Dataset({
    features: ['messages'],
    num_rows: 2539
})

To format our dataset, we will apply our version of the GPT OSS prompt

In [9]:
# ============================================
# FORMAT DATASET FOR TRAINING
# ============================================
# Format dataset để phù hợp với tokenizer GPT-OSS
# Dataset đã được convert sang format GPT-OSS Harmony ở cell trước

def formatting_prompts_func(examples):
    """
    Format messages thành text để training
    """
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Apply formatting
print("Đang format dataset cho training...")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Đã format {len(dataset)} mẫu dữ liệu")

# Show statistics
text_lengths = [len(text.split()) for text in dataset["text"]]
import numpy as np
print(f"\nThống kê độ dài text:")
print(f"  - Trung bình: {np.mean(text_lengths):.1f} từ")
print(f"  - Median: {np.median(text_lengths):.1f} từ")
print(f"  - Min: {np.min(text_lengths)} từ")
print(f"  - Max: {np.max(text_lengths)} từ")
print(f"  - 99th percentile: {np.percentile(text_lengths, 99):.1f} từ")

Đang format dataset cho training...


Map: 100%|██████████| 2539/2539 [00:00<00:00, 14730.68 examples/s]

Đã format 2539 mẫu dữ liệu

Thống kê độ dài text:
  - Trung bình: 514.4 từ


  - Median: 496.0 từ
  - Min: 206 từ
  - Max: 1539 từ
  - 99th percentile: 989.5 từ


Let's take a look at the dataset, and check what the 1st example shows

In [10]:
print(dataset[0]['text'])

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-06

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

# Instructions

reasoning language: Vietnamese

Bạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành. Câu trả lời phải chính xác, có căn cứ pháp lý rõ ràng và dễ hiểu.<|end|><|start|>user<|message|>Hội đồng giải thể doanh nghiệp do Nhà nước nắm giữ 100% vốn điều lệ có được quyền sử dụng con dấu của doanh nghiệp hay không?<|end|><|start|>assistant<|message|><|channel|>analysis<|message|>Câu hỏi này liên quan đến pháp luật Việt Nam. Tôi cần phân tích để đưa ra câu trả lời chính xác dựa trên các văn bản quy phạm pháp luật hiện hành.

Căn cứ p

What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Train the model
Now let's train our model for 5 epochs. Training progress will be logged to WandB for monitoring.

In [11]:
# ============================================
# WANDB LOGIN & SETUP
# ============================================
import wandb

# Login to WandB (uncomment and set your API key if needed)
# wandb.login(key="your_wandb_api_key_here")

# Or use environment variable: export WANDB_API_KEY=your_key
# Or run: wandb login in terminal

print("WandB setup complete!")


WandB setup complete!


In [12]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, # Train for 5 epochs
        # max_steps removed - use num_train_epochs instead
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", # Log to WandB
        run_name = "gpt-oss-20b-legal-vietnam-5epochs", # WandB run name
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=20): 100%|██████████| 2539/2539 [00:06<00:00, 398.61 examples/s]


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes and lower loss as well!

In [13]:
from unsloth.chat_templates import train_on_responses_only

# ============================================
# TRAIN ON RESPONSES ONLY
# ============================================
# Với GPT-OSS Harmony format:
# - instruction_part: Phần user message (câu hỏi)
# - response_part: Phần assistant response (bao gồm cả analysis và final channels)
# Format thực tế: <|start|>assistant<|message|><|channel|>analysis<|message|>...<|channel|>final<|message|>...

gpt_oss_kwargs = dict(
    instruction_part = "<|start|>user<|message|>", 
    response_part = "<|start|>assistant<|message|>",  # Train trên toàn bộ assistant response
    tokenizer = tokenizer,  # Cần tokenizer để tìm pattern chính xác
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

Map (num_proc=20): 100%|██████████| 2539/2539 [00:01<00:00, 2516.81 examples/s]


Let's verify masking the instruction part is done! Let's print the 100th row again.

In [14]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2025-11-06\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions\n\n# Instructions\n\nreasoning language: Vietnamese\n\nBạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành. Câu trả lời phải chính xác, có căn cứ pháp lý rõ ràng và dễ hiểu.<|end|><|start|>user<|message|>Người thành lập doanh nghiệp có được ký hợp đồng trước khi đăng ký doanh nghiệp không?<|end|><|start|>assistant<|message|><|channel|>analysis<|message|>Câu hỏi này liên quan đến pháp luật Việt Nam. Tôi cần phân tích để đưa ra câu trả lời chính xác dựa trên các văn bản quy phạm pháp luật hiện hành.\n\nCăn cứ pháp lý: Điều 18 Luật Do

Now let's print the masked out example - you should see only the answer is present:

In [15]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                              <|channel|>analysis<|message|>Câu hỏi này liên quan đến pháp luật Việt Nam. Tôi cần phân tích để đưa ra câu trả lời chính xác dựa trên các văn bản quy phạm pháp luật hiện hành.\n\nCăn cứ pháp lý: Điều 18 Luật Doanh Nghiệp 2020\n\nĐây là câu hỏi loại reasoning, tôi sẽ trả lời dựa trên các quy định cụ thể.\n<|channel|>final<|message|>Hợp đồng trước đăng ký doanh nghiệp được quy định tại Điều 18 Luật Doanh nghiệp 2020 như sau:\nHợp đồng trước đăng ký doanh nghiệp\n1. Người thành lập doanh nghiệp được ký hợp đồng phục vụ cho việc thành lập và hoạt động của doanh nghiệp trước và trong quá trình đăng ký doanh nghiệp.\n2. Trường hợp được cấp Giấy chứng nhận đăng ký doanh nghiệp, doanh nghiệp phải tiếp tục thực hiện quyền và nghĩa vụ phát sinh từ hợp đồng đã ký kết quy định tại khoản 1 Điều này và các bên phải thực hiện v

In [16]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5060 Ti. Max memory = 15.444 GB.
12.873 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [17]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,539 | Num Epochs = 5 | Total steps = 3,175
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,981,312 of 20,918,738,496 (0.02% trained)
wandb: Currently logged in as: nhotin911 (nhotin911-FPT University) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,12.924100
2,12.760500
3,13.906900
4,11.636800
5,5.403900
6,3.455900
7,2.073200
8,1.513300
9,1.506800
10,1.315400


In [18]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

16448.9041 seconds used for training.
274.15 minutes used for training.
Peak reserved memory = 14.512 GB.
Peak reserved memory for training = 1.639 GB.
Peak reserved memory % of max memory = 93.965 %.
Peak reserved memory for training % of max memory = 10.613 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [19]:
# ============================================
# INFERENCE - Test model với câu hỏi pháp luật Việt Nam
# ============================================
messages = [
    {
        "role": "developer",
        "content": (
            "# Instructions\n\nreasoning language: Vietnamese\n\n"
            "Bạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. "
            "Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành."
        )
    },
    {
        "role": "user", 
        "content": "Doanh nghiệp có được quyền tự quyết định về việc sử dụng con dấu hay không?"
    },
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # Có thể thay đổi: "low", "medium", "high"
).to("cuda")

from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 256, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-07

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

# Instructions

reasoning language: Vietnamese

Bạn là một trợ lý AI chuyên về tư vấn pháp luật Việt Nam. Hãy phân tích và trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành.<|end|><|start|>user<|message|>Doanh nghiệp có được quyền tự quyết định về việc sử dụng con dấu hay không?<|end|><|start|>assistant<|channel|>analysis<|message|>Câu hỏi này liên quan đến pháp luật Việt Nam. Tôi cần phân tích để đưa ra câu trả lời phù hợp dựa trên các văn bản quy phạm pháp luật hiện hành.

Căn cứ theo luật hiện hành: Điều 29 Luật Doanh Nghiệp 2020

Đây là câu hỏi loại query, tôi sẽ trả lời dựa trên các quy định cụ thể.
<|cha

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** Currently finetunes can only be loaded via Unsloth in the meantime - we're working on vLLM and GGUF exporting!

In [20]:
model.save_pretrained("finetuned_model")
# model.push_to_hub("hf_username/finetuned_model", token = "hf_...") # Save to HF

To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [21]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "finetuned_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 1024,
        dtype = None,
        load_in_4bit = True,
    )

messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-11-07

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are a helpful assistant that can solve mathematical problems.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the equation \(x^5 + 3x^4 - 10 = 3\). This is a polynomial equation of degree 5. We will rewrite it as:
\[ x^5 + 3x^4 - 10 = 3 \]

Let's subtract 


### Saving to float16 for VLLM or mxfp4

We also support saving to `float16` or `mxfp4` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [22]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "mxfp4")
if False: model.push_to_hub_merged("repo_id/repo_name", tokenizer, token = "hf...", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/gpt-oss-finetune", tokenizer, save_method = "merged_16bit", token = "")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i>
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
